# Colab — Hard negative mining (pipeline clásico)

Re-corre el detector sobre las imágenes train, identifica falsos positivos, los añade al training set como negativos extra y re-entrena la SVM. Repite por `hard_negative.rounds` rondas (default 2).

**Tarda** comparable a `colab_build_features.ipynb` × nº rondas.

## Resume robusto

- **Por ronda**: `.hardneg_state.json` en Drive guarda `completed_rounds`. Si Colab cae después de la ronda 1, al relanzar empieza por la ronda 2.
- **Dentro de ronda**: checkpoint parcial cada 100 imgs en `classical_features.hardneg_partial.npz`, también en Drive vía symlink + atómico.

Resultado: imposible perder más de 100 imgs por corte de Colab.

## Prerequisitos

En `MyDrive/grocery-detection/processed/`:
- `train.json`
- `codebook.pkl`
- `classical_features.npz`  (de colab_build_features.ipynb)
- `classical_svm.pkl`        (de colab_train_svm.ipynb)

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/arturmoret/GroceryTracker.git"
REPO_DIR = "/content/repo"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)

sys.path.insert(0, REPO_DIR + "/scripts")
sys.path.insert(0, REPO_DIR + "/src")
os.chdir(REPO_DIR)

from colab_helper import (
    mount_drive, install_deps, setup_dataset,
    link_processed_to_drive, run_script,
)
print("helpers cargados")

In [ ]:
mount_drive()
install_deps()

In [ ]:
setup_dataset()  # D2S de Drive a /content

In [ ]:
link_processed_to_drive()  # data/processed/ ↔ Drive/grocery-detection/processed/

## Mining + refit

Para forzar empezar desde la ronda 1, descomenta el `--reset`.

In [ ]:
run_script("scripts/run_classical_hard_neg.py")
# run_script("scripts/run_classical_hard_neg.py", "--reset")

## Listo

`classical_features.npz`, `classical_svm.pkl` y `.hardneg_state.json` actualizados en Drive. Sin sync manual. Siguiente: `colab_infer.ipynb`.